# 03 - Model vs NWS flood warnings

Benchmarks the trained GOES/GLM model against **NWS Flash-Flood + Areal-Flood WARNINGS** -
the operational forecaster baseline. Both are rasterized to the 50 km grid on the **test**
days (2025 odd months) and scored against the **observed-flood labels** (Groundsource union
NCEI storm events - the same `y` the model was trained on).

Note on fairness: NWS warnings are same-day, short-lead nowcasts, while our model predicts
day D from **D-1** GOES (a ~1-day lead). So this is "who better flags the cells that flood",
not a like-for-like lead comparison - the model is doing a strictly harder (longer-lead) task.

Sections: (1) model predictions, (2) warnings -> 50 km grid, (3) metrics table
(model & warnings vs observed), (4) who catches the observed floods, (5) per-day maps,
(6) aggregate spatial view, (7) metric bar chart.

## 0. Setup

In [ ]:
import pickle
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, precision_recall_curve
from torch.utils.data import DataLoader

ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
MODEL_DIR = ROOT / "notebooks" / "model"
OUT_DIR = MODEL_DIR / "outputs"
sys.path.insert(0, str(MODEL_DIR))
sys.path.insert(0, str(MODEL_DIR / "trainers"))

from config import STATES_GEOJSON, UNIFIED_PARQUET, build_grid_cells
import cnn3d, resnet3d, cnn_attn, convgru, convgru_attn   # noqa: E401  (deep)
import logreg, xgb                          # noqa: E401  (tabular trainers)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DEEP = {"cnn3d": cnn3d, "resnet3d": resnet3d, "cnn_attn": cnn_attn,
        "convgru": convgru, "convgru_attn": convgru_attn}
TAB = {"logreg": logreg, "xgb": xgb}

_, GRID_R, GRID_C, land = build_grid_cells()
CACHE_DIR = cnn3d.CACHE_DIR
tr_days, va_days, te_days = cnn3d.load_splits()


def base_rate(days):
    y = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in days])
    return float(y[:, land].mean())


print(f"device {DEVICE} | grid {GRID_R}x{GRID_C} | land {int(land.sum())} cells")
print(f"test days: {len(te_days)} (2025 odd months)  base rate {base_rate(te_days):.4f}")

# grid polygons + state outlines for maps + rasterizing warnings
cells_gdf, _, _, _ = build_grid_cells()
cells_albers = cells_gdf.to_crs(5070)
RR, CC = cells_albers["R"].to_numpy(), cells_albers["C"].to_numpy()
states = gpd.read_file(STATES_GEOJSON)
states = states[~states["name"].isin(["Alaska", "Hawaii", "Puerto Rico"])].to_crs(5070)

## 1. Model predictions

Reload the chosen model, predict per-cell flood probability on val + test, pick the best-F1
threshold on **val**, and binarize the **test** predictions at that threshold (the honest
operating point). Deep models normalize inputs internally (saved buffers); tabular models use
`predict_grids`.

In [ ]:
# a single model name, or a dict {name: weight} to ENSEMBLE (weighted-avg probability)
MODEL = {"resnet3d": 0.5, "convgru_attn": 0.5}
# examples:  "convgru_attn"  |  {"resnet3d": 0.4, "convgru_attn": 0.4, "xgb": 0.2}

_PCACHE = {}


@torch.no_grad()
def _infer_one(name, days):
    """(probs, trues) for one deep or tabular model."""
    if name in DEEP:
        mod = DEEP[name]
        net = mod.FloodNet().to(DEVICE)
        net.load_state_dict(torch.load(OUT_DIR / f"{name}.pt", map_location=DEVICE))
        net.eval()
        probs, trues = [], []
        loader = DataLoader(mod.FeatureCache(days), batch_size=16, num_workers=8)
        for seq, summ, t, y in loader:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                pr = torch.sigmoid(net(seq.to(DEVICE).float(), summ.to(DEVICE).float(),
                                       t.to(DEVICE).float())).squeeze(1)
            probs.append(pr.float().cpu().numpy())
            trues.append(y.numpy())
        del net
        torch.cuda.empty_cache()
        return np.concatenate(probs), np.concatenate(trues)
    payload = pickle.load(open(OUT_DIR / f"{name}.pkl", "rb"))
    return TAB[name].predict_grids(payload, days, land)


def _component(name, days, tag):
    key = (name, tag)
    if key not in _PCACHE:
        _PCACHE[key] = _infer_one(name, days)
    return _PCACHE[key]


def infer_probs(spec, days, tag):
    """spec = name or {name: weight}; returns weighted-average (probs, trues)."""
    if isinstance(spec, str):
        return _component(spec, days, tag)
    wsum = sum(spec.values())
    probs, trues = None, None
    for name, w in spec.items():
        pr, y = _component(name, days, tag)
        probs = (w / wsum) * pr if probs is None else probs + (w / wsum) * pr
        trues = y
    return probs, trues


def spec_name(spec):
    if isinstance(spec, str):
        return spec
    return " + ".join(f"{w:g}*{n}" for n, w in spec.items())


def _exists(spec):
    names = [spec] if isinstance(spec, str) else list(spec)
    return all((OUT_DIR / f"{n}.pt").exists() or (OUT_DIR / f"{n}.pkl").exists()
               for n in names)


assert _exists(MODEL), f"{MODEL} not fully trained yet"
MODEL_NAME = spec_name(MODEL)

vp, _ = infer_probs(MODEL, va_days, "val")
probs, Y = infer_probs(MODEL, te_days, "test")          # test probabilities + observed labels

yv = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in va_days])
pr, rc, thr = precision_recall_curve(yv[:, land].ravel(), vp[:, land].ravel())
f1c = 2 * pr * rc / (pr + rc + 1e-9)
THR = float(thr[np.argmax(f1c[:-1])])
M = (probs >= THR).astype(np.float32)                   # binarized prediction (test)
ap = average_precision_score(Y[:, land].ravel(), probs[:, land].ravel())
print(f"model: {MODEL_NAME}")
print(f"test AUPRC {ap:.4f}  |  val-F1 threshold {THR:.3f}  |  "
      f"flagged cells/day {M[:, land].sum() / len(te_days):.1f}")

## 2. NWS warnings -> 50 km grid

A cell is **warned** on CDT day D if an NWS Flash-Flood (FF) or Areal-Flood (FA) *warning*
polygon is active during that CDT day and intersects the cell. Warning issue/expire are UTC;
we shift by -5 h to CDT (matching the label construction) and rasterize onto the same grid,
for the test days only.

In [ ]:
CDT = pd.Timedelta(hours=5)
te_ts = pd.to_datetime(te_days, format="%Y%m%d")
te_set = set(te_ts)

u = gpd.read_parquet(UNIFIED_PARQUET)
w = u[u["source"].isin(["ff_warning", "fa_warning"])].copy()
w["issue_cdt"] = w["issue_date"] - CDT
w["expire_cdt"] = w["expire_date"] - CDT
# keep warnings that could touch a test CDT day (cheap pre-filter)
w = w[(w["expire_cdt"] >= te_ts.min()) & (w["issue_cdt"] <= te_ts.max() + pd.Timedelta(days=1))]


def cdt_days(issue, expire):
    d = pd.date_range(issue.normalize(), expire.normalize(), freq="D")
    return d if len(d) <= 5 else d[:1]


w["day"] = [cdt_days(i, e) for i, e in zip(w["issue_cdt"], w["expire_cdt"])]
ev = w.explode("day", ignore_index=True)
ev = ev[ev["day"].isin(te_set)]

j = gpd.sjoin(cells_gdf, ev[["day", "geometry"]], predicate="intersects")
warn_map = {}
for day, g in j.groupby("day"):
    a = np.zeros((GRID_R, GRID_C), np.float32)
    a[g["R"], g["C"]] = 1.0
    warn_map[day.strftime("%Y%m%d")] = a

W = np.stack([warn_map.get(d, np.zeros((GRID_R, GRID_C), np.float32)) for d in te_days])
print(f"warnings rasterized: {len(ev):,} warning-days -> "
      f"{sum(v.sum() > 0 for v in warn_map.values())}/{len(te_days)} test days have >=1 warning")
print(f"warned land cells/day: {W[:, land].sum() / len(te_days):.1f}  "
      f"(observed floods/day: {Y[:, land].sum() / len(te_days):.1f})")

## 3. Metrics - model & warnings vs observed floods

Both predictions scored against the observed labels on land cells: precision, recall, F1, CSI
(exact and 1-grid-tolerant). The model additionally has AUPRC (probabilistic); warnings are
binary. `xbase` = AUPRC / base rate.

In [ ]:
def _dilate(m):
    o = m.copy()
    o[:-1, :] |= m[1:, :]; o[1:, :] |= m[:-1, :]
    o[:, :-1] |= m[:, 1:]; o[:, 1:] |= m[:, :-1]
    o[:-1, :-1] |= m[1:, 1:]; o[1:, 1:] |= m[:-1, :-1]
    o[:-1, 1:] |= m[1:, :-1]; o[1:, :-1] |= m[:-1, 1:]
    return o


def binary_metrics(pred, y):
    """P/R/F1/CSI exact + 1-grid for a binary prediction (N,R,C) vs labels."""
    pb = pred[:, land].ravel().astype(int)
    t = y[:, land].ravel().astype(int)
    tp = int((pb * t).sum()); fp = int((pb * (1 - t)).sum()); fn = int(((1 - pb) * t).sum())
    prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
    f1 = 2 * prec * rec / (prec + rec + 1e-9); csi = tp / (tp + fn + fp + 1e-9)
    h = m1 = fa = 0
    for i in range(len(pred)):
        yt = (y[i] > 0.5) & land
        yp = (pred[i] > 0.5) & land
        h += int((yt & (_dilate(yp) & land)).sum())
        m1 += int((yt & ~(_dilate(yp) & land)).sum())
        fa += int((yp & ~(_dilate(yt) & land)).sum())
    p1 = h / (h + fa + 1e-9); r1 = h / (h + m1 + 1e-9)
    return dict(P=prec, R=rec, F1=f1, CSI=csi,
                F1_1=2 * p1 * r1 / (p1 + r1 + 1e-9), CSI1=h / (h + m1 + fa + 1e-9))


base = base_rate(te_days)
rows = []
mm = binary_metrics(M, Y)
mm["AUPRC"] = average_precision_score(Y[:, land].ravel(), probs[:, land].ravel())
mm["xbase"] = mm["AUPRC"] / base
rows.append(dict(predictor=f"{MODEL_NAME} (D-1 GOES)", **mm))
wm = binary_metrics(W, Y)
wm["AUPRC"] = float("nan"); wm["xbase"] = float("nan")
rows.append(dict(predictor="NWS warnings", **wm))

tbl = pd.DataFrame(rows).set_index("predictor")[
    ["AUPRC", "xbase", "P", "R", "F1", "CSI", "F1_1", "CSI1"]]
print(f"test base rate {base:.4f}  ({len(te_days)} days)\n")
print(tbl.round(3).to_string())

## 3b. Single models vs ensembles

Evaluate each single model and the two candidate ensembles (weighted-average probability) against observed floods on the test set - each thresholded on val. Shows whether ensembling beats the best single model.

In [ ]:
CANDIDATES = ["convgru_attn", "convgru", "resnet3d", "cnn3d", "xgb",
              {"resnet3d": 0.5, "convgru": 0.5},
              {"resnet3d": 0.5, "convgru_attn": 0.5},
              {"convgru": 0.5, "convgru_attn": 0.5},
              {"resnet3d": 0.4, "convgru_attn": 0.4, "xgb": 0.2}]

yv_land = yv[:, land].ravel()
rows = []
for spec in CANDIDATES:
    if not _exists(spec):
        continue
    vpp, _ = infer_probs(spec, va_days, "val")
    pp, yy = infer_probs(spec, te_days, "test")
    prc, rcc, thc = precision_recall_curve(yv_land, vpp[:, land].ravel())
    fc = 2 * prc * rcc / (prc + rcc + 1e-9)
    th = float(thc[np.argmax(fc[:-1])])
    apc = average_precision_score(yy[:, land].ravel(), pp[:, land].ravel())
    bm = binary_metrics((pp >= th).astype(np.float32), yy)
    rows.append(dict(model=spec_name(spec), AUPRC=apc, xbase=apc / base,
                     F1=bm["F1"], CSI=bm["CSI"], F1_1=bm["F1_1"], CSI1=bm["CSI1"]))

cand = pd.DataFrame(rows).set_index("model").sort_values("AUPRC", ascending=False)
print("test set - single models vs ensembles (sorted by AUPRC):\n")
print(cand.round(3).to_string())
_bst = cand.index[0]
print(f"\n>>> best by AUPRC: {_bst}  ({cand.iloc[0]['AUPRC']:.4f}, {cand.iloc[0]['xbase']:.1f}x)")

## 4. Who catches the observed floods?

Of every observed flood cell-day on the test set, how many are flagged by the **model only**,
**NWS only**, **both**, or **neither** - and the same for false alarms (flagged but no observed
flood). This shows whether the model adds coverage beyond the operational warnings.

In [ ]:
ml = M[:, land].astype(bool).ravel()
wl = W[:, land].astype(bool).ravel()
yl = Y[:, land].astype(bool).ravel()

pos = yl.sum()
both = int((ml & wl & yl).sum())
mo = int((ml & ~wl & yl).sum())
wo = int((~ml & wl & yl).sum())
neither = int((~ml & ~wl & yl).sum())
print(f"observed flood cell-days on test: {pos}")
print(f"  caught by BOTH        : {both:6d}  ({both / pos:5.1%})")
print(f"  caught by MODEL only  : {mo:6d}  ({mo / pos:5.1%})")
print(f"  caught by NWS only    : {wo:6d}  ({wo / pos:5.1%})")
print(f"  MISSED by both        : {neither:6d}  ({neither / pos:5.1%})")
print(f"\nrecall  model {(both + mo) / pos:.1%}   NWS {(both + wo) / pos:.1%}")
inter = int((ml & wl).sum()); union = int((ml | wl).sum())
print(f"model-vs-NWS footprint IoU: {inter / (union + 1e-9):.3f}  "
      f"(model flags {int(ml.sum())}, NWS flags {int(wl.sum())} cell-days)")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.barh(["both", "model only", "NWS only", "missed by both"],
        [both, mo, wo, neither], color=["#4c956c", "#e09f3e", "#6a4c93", "#c1121f"])
ax.set_xlabel("observed flood cell-days")
ax.set_title(f"Coverage of observed floods - {MODEL_NAME} vs NWS warnings (test)")
for i, v in enumerate([both, mo, wo, neither]):
    ax.text(v, i, f" {v / pos:.0%}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

## 5. Per-day maps

Random test days: observed floods, the model's binary prediction (@ val threshold), and the
NWS warning footprint. Change `SEED`/`N_SHOW`.

In [ ]:
SEED = 44
N_SHOW = 4


def _draw(ax, values, cmap, title=None, ylabel=None):
    gdf = cells_albers.copy(); gdf["v"] = values[RR, CC]
    gdf.boundary.plot(ax=ax, color="white", lw=0.1, zorder=2)
    gdf.plot(column="v", cmap=cmap, ax=ax, zorder=1, edgecolor="none", vmin=0, vmax=1)
    states.boundary.plot(ax=ax, color="0.5", lw=0.5, zorder=3)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    if title:
        ax.set_title(title, fontsize=11, fontweight="bold")
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=12, fontweight="bold")


rng = np.random.default_rng(SEED)
sel = sorted(rng.choice(len(te_days), size=min(N_SHOW, len(te_days)), replace=False))
rows = [("Observed", Y, "Greens"), (f"{MODEL_NAME}", M, "Oranges"), ("NWS warnings", W, "Purples")]
fig, axes = plt.subplots(len(rows), len(sel), figsize=(3.4 * len(sel), 3.1 * len(rows)),
                         squeeze=False)
for c, i in enumerate(sel):
    ny = int((Y[i][land] > 0).sum())
    for r, (label, arr, cmap) in enumerate(rows):
        _draw(axes[r, c], arr[i], cmap,
              title=f"{te_days[i]}  ({ny} floods)" if r == 0 else None,
              ylabel=label if c == 0 else None)
fig.suptitle(f"Test days (seed {SEED}) - observed vs {MODEL_NAME} vs NWS warnings",
             fontsize=14, fontweight="bold", y=1.002)
plt.tight_layout(); plt.show()

## 6. Aggregate spatial view

Summed over all test days: where floods were observed, where the model fired, and where NWS
warned - so persistent hits / gaps stand out geographically.

In [ ]:
agg = [("Observed flood-days", Y.sum(0)), (f"{MODEL_NAME} flagged-days", M.sum(0)),
       ("NWS warned-days", W.sum(0))]
mx = max(float(a[land].max()) for _, a in agg) or 1.0
fig, axes = plt.subplots(1, 3, figsize=(4.4 * 3, 4.2))
for ax, (title, a) in zip(axes, agg):
    gdf = cells_albers.copy(); gdf["v"] = np.where(land, a, np.nan)[RR, CC]
    gdf.plot(column="v", cmap="magma_r", ax=ax, vmin=0, vmax=mx, edgecolor="none")
    states.boundary.plot(ax=ax, color="0.5", lw=0.5)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
fig.colorbar(plt.cm.ScalarMappable(cmap="magma_r",
             norm=plt.Normalize(0, mx)), ax=axes, fraction=0.02, pad=0.01, label="days")
fig.suptitle("Aggregate over test days (0-{:.0f})".format(mx), fontsize=13, fontweight="bold")
plt.show()

## 7. Metric bar chart

Side-by-side precision / recall / F1 / CSI (exact + 1-grid) for the model vs NWS warnings,
both against observed floods.

In [ ]:
keys = ["P", "R", "F1", "CSI", "F1_1", "CSI1"]
labels = ["Prec", "Recall", "F1", "CSI", "F1@1", "CSI@1"]
xm = np.arange(len(keys))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(xm - 0.2, [mm[k] for k in keys], 0.4, label=f"{MODEL_NAME} (D-1 GOES)", color="#e09f3e")
ax.bar(xm + 0.2, [wm[k] for k in keys], 0.4, label="NWS warnings", color="#6a4c93")
ax.set_xticks(xm); ax.set_xticklabels(labels)
ax.set_ylabel("score (vs observed floods)")
ax.set_title(f"{MODEL_NAME} vs NWS warnings on the test set")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()